# 03 — Market, trajectory, availability (Phase 3)

Plan: `docs/superpowers/plans/2026-08-29-phase3-market-trajectory-availability.md`. Inputs are the
Phase 1 tables and the Phase 2 artifacts (`models/phase2_*.json`). Every fit is leave-future-out.

In [1]:
import json

import numpy as np
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 300)

from scout import config
from scout.data import reep, understat
from scout.data import transfermarkt as tm_loader
from scout.identity import build_team_lineage, load_overrides
from scout.panel import identity, stints, team_season

LEAGUE_TO_COMP = {league: comp for comp, league in config.BIG5.items()}
COMPS = list(config.BIG5) + list(config.FEEDERS)

## Step 1 — The market target and its sample

What `value_july` looks like (log scale) for players with a Phase 2 contribution row; coverage by
league-season; how value relates to contribution, age, minutes, league and club strength in the raw
data. Candidates: value at 1 July of the stats season vs at 1 July of the next season; club tier as
the club's expected points, its squad-value rank, or its Elo on 1 July.

In [2]:
# Phase 2 contribution rows (Understat ids) -> Transfermarkt ids -> stints with the two valuations
contrib = pd.DataFrame(json.load(open(config.MODELS / "phase2_contribution.json")))
tm_panel = tm_loader.load_player_club_seasons(COMPS, list(config.SEASONS))
tm_clubs = tm_panel[["club_id", "club_name", "competition_id"]].drop_duplicates()
us = understat.load("player_season")
us["competition_id"] = us.league.map(LEAGUE_TO_COMP)
lineage = build_team_lineage(tm_clubs, {"understat": us[["competition_id", "team"]].drop_duplicates().rename(columns={"team": "team_name"})}, load_overrides("teams"))
us_ids = identity.resolve_provider("understat", us, identity.transfermarkt_side(tm_panel), lineage, reep.load_people()).drop_duplicates("provider_id").set_index("provider_id").tm_player_id
contrib["tm_player_id"] = contrib.player_id.astype(int).astype(str).map(us_ids)

st = stints.build(COMPS, list(config.SEASONS))
st["tm_player_id"] = st.tm_player_id.astype(str)
season_value = st.sort_values("minutes", ascending=False).drop_duplicates(["tm_player_id", "season"])[["tm_player_id", "season", "club_id", "competition_id", "value_july", "value_age_days_july"]]
players = tm_loader.load_table("players")[["player_id", "date_of_birth"]]
players["tm_player_id"] = players.player_id.astype(str)

rows = contrib.dropna(subset=["tm_player_id"]).merge(season_value, on=["tm_player_id", "season"], how="left", suffixes=("", "_tm"))
rows = rows.merge(players[["tm_player_id", "date_of_birth"]], on="tm_player_id", how="left")
rows["age"] = rows.season + 1 - pd.to_datetime(rows.date_of_birth).dt.year
nxt = season_value.assign(season=season_value.season - 1)[["tm_player_id", "season", "value_july"]].rename(columns={"value_july": "value_next_july"})
rows = rows.merge(nxt, on=["tm_player_id", "season"], how="left")

print(len(rows), "contribution rows |", f"with a Transfermarkt id {rows.tm_player_id.notna().mean():.1%}", f"| with value_july {rows.value_july.notna().mean():.1%}", f"| with next July's value {rows.value_next_july.notna().mean():.1%}")
print("value coverage by league-season:")
print(rows.groupby(["competition_id", "season"]).value_july.apply(lambda c: c.notna().mean()).unstack("season").round(2).to_string())

21427 contribution rows | with a Transfermarkt id 100.0% | with value_july 99.0% | with next July's value 78.0%
value coverage by league-season:
season          2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
competition_id                                                                        
ES1             0.98  0.98  0.97  0.99  0.99  1.00  0.99  0.99  0.99  0.99  0.99  1.00
FR1             0.97  0.95  0.97  0.98  0.96  0.97  0.97  0.99  0.99  0.99  0.98  0.98
GB1             0.99  0.99  0.99  0.99  0.99  0.97  0.99  1.00  1.00  0.99  1.00  1.00
IT1             1.00  1.00  1.00  1.00  1.00  0.99  1.00  1.00  1.00  1.00  1.00  1.00
L1              1.00  1.00  0.99  0.99  1.00  1.00  0.98  1.00  1.00  0.99  0.99  0.99


In [3]:
# The target on a log scale, by role and season
valued = rows.dropna(subset=["value_july"]).copy()
valued["log_value"] = np.log10(valued.value_july)

print("log10 value_july by role:")
print(valued.groupby("role").log_value.describe(percentiles=[0.1, 0.5, 0.9]).round(2)[["count", "mean", "std", "10%", "50%", "90%"]].to_string())
print("\nmedian value (M€) by season:", valued.groupby("season").value_july.median().div(1e6).round(2).to_dict())

log10 value_july by role:
       count  mean   std   10%   50%   90%
role                                      
CB    4503.0  6.65  0.56  5.90  6.65  7.40
CM    4991.0  6.75  0.57  6.00  6.78  7.48
FB    4031.0  6.60  0.54  5.90  6.60  7.30
ST    2919.0  6.83  0.56  6.18  6.85  7.58
W     4770.0  6.82  0.57  6.08  6.85  7.54

median value (M€) by season: {2014: 3.0, 2015: 3.0, 2016: 3.5, 2017: 4.0, 2018: 5.0, 2019: 8.0, 2020: 6.5, 2021: 7.0, 2022: 7.0, 2023: 7.0, 2024: 8.0, 2025: 8.0}


In [4]:
# Raw relationships: binned means of log value against each candidate feature
def binned(frame, col, bins, label):
    cut = pd.cut(frame[col], bins)
    table = frame.groupby(cut, observed=True).log_value.agg(["mean", "size"]).round(2)
    print(f"\n{label}:")
    print(table.T.to_string())


binned(valued, "point", [0, 0.05, 0.1, 0.2, 0.3, 0.45, 0.6, 0.8, 1.2, 3], "contribution (shrunk point, per 90)")
binned(valued, "age", [16, 20, 22, 24, 26, 28, 30, 32, 34, 45], "age")
binned(valued, "minutes", [600, 900, 1500, 2000, 2500, 3500], "minutes")
print("\nleague:", valued.groupby("competition_id").log_value.mean().round(2).to_dict())
print("role × contribution tercile:")
valued["contrib_tercile"] = valued.groupby("role").point.transform(lambda x: pd.qcut(x, 3, labels=["low", "mid", "high"]))
print(valued.pivot_table(index="role", columns="contrib_tercile", values="log_value", aggfunc="mean", observed=True).round(2).to_string())


contribution (shrunk point, per 90):
point  (0.0, 0.05]  (0.05, 0.1]  (0.1, 0.2]  (0.2, 0.3]  (0.3, 0.45]  (0.45, 0.6]  (0.6, 0.8]  (0.8, 1.2]
mean          6.45         6.63        6.68         6.8         6.78         6.93        7.17         7.7
size       2149.00      5077.00     4670.00      2707.0      3096.00      2430.00      948.00       114.0

age:
age   (16, 20]  (20, 22]  (22, 24]  (24, 26]  (26, 28]  (28, 30]  (30, 32]  (32, 34]  (34, 45]
mean      6.43      6.64      6.78      6.83      6.84      6.82       6.7      6.49      6.13
size    519.00   1858.00   3350.00   3920.00   3815.00   3156.00    2362.0   1356.00    870.00

minutes:
minutes  (600, 900]  (900, 1500]  (1500, 2000]  (2000, 2500]  (2500, 3500]
mean           6.62         6.69          6.74           6.8          6.87
size        4220.00      6480.00       4226.00        3250.0       3024.00

league: {'ES1': 6.68, 'FR1': 6.51, 'GB1': 7.06, 'IT1': 6.68, 'L1': 6.71}
role × contribution tercile:
contrib_tercile

In [5]:
# Club tier candidates: expected points (Understat), squad value rank (Transfermarkt), Elo on 1 July (ClubElo)
from scout.data import clubelo
from scout.panel import elo as elo_panel

ts = team_season.build()
ts["competition_id"] = ts.league.map(LEAGUE_TO_COMP)
team_club = us[["competition_id", "team", "team_id"]].drop_duplicates().merge(lineage[["competition_id", "team_name", "club_id"]].rename(columns={"team_name": "team"}), on=["competition_id", "team"])
ts = ts.merge(team_club[["competition_id", "team_id", "club_id"]], on=["competition_id", "team_id"])
club_strength = ts[["competition_id", "season", "club_id", "expected_points_for"]]

squad_value = season_value.groupby(["competition_id", "season", "club_id"]).value_july.sum().rename("squad_value").reset_index()
squad_value["squad_rank"] = squad_value.groupby(["competition_id", "season"]).squad_value.rank(ascending=False)

elo_names = elo_panel.club_elo_names(COMPS, list(config.SEASONS))
club_elo = []
for (comp, season, club_id), _ in squad_value.groupby(["competition_id", "season", "club_id"]):
    name = elo_names.get(club_id)
    if name is None:
        continue
    e = elo_panel.elo_on_dates(clubelo.fetch_club(name), pd.Series([f"{season}-07-01"]))[0]
    club_elo.append((comp, season, club_id, e))
club_elo = pd.DataFrame(club_elo, columns=["competition_id", "season", "club_id", "club_elo_july"])

tiers = club_strength.merge(squad_value, on=["competition_id", "season", "club_id"], how="outer").merge(club_elo, on=["competition_id", "season", "club_id"], how="outer")
v = valued.merge(tiers, on=["competition_id", "season", "club_id"], how="left")
print("club-tier coverage among valued rows:", {c: f"{v[c].notna().mean():.1%}" for c in ["expected_points_for", "squad_rank", "club_elo_july"]})

# which tier explains most of the residual after contribution and age (within role and season)?
import statsmodels.formula.api as smf

base = smf.ols("log_value ~ C(role) * (point + age + I(age**2)) + C(competition_id) + C(season)", data=v).fit()
v["resid"] = base.resid
print(f"\nbase model R² (contribution, age, role, league, season): {base.rsquared:.3f}")
for c in ["expected_points_for", "squad_rank", "club_elo_july"]:
    sub = v.dropna(subset=[c])
    r = np.corrcoef(sub.resid, sub[c])[0, 1]
    extra = smf.ols(f"resid ~ {c}", data=sub).fit().rsquared
    print(f"  {c}: r with residual {r:+.3f} | residual variance explained {extra:.3f} (n={len(sub)})")

club-tier coverage among valued rows: {'expected_points_for': '99.3%', 'squad_rank': '99.3%', 'club_elo_july': '99.3%'}



base model R² (contribution, age, role, league, season): 0.375
  expected_points_for: r with residual +nan | residual variance explained 0.225 (n=21074)
  squad_rank: r with residual +nan | residual variance explained 0.383 (n=21074)
  club_elo_july: r with residual +nan | residual variance explained 0.279 (n=21074)


### Step 1 note — target and sample, from the outputs above

**Sample.** 21,427 Phase 2 contribution rows, 100% with a Transfermarkt id, 99.0% with a
valuation at 1 July of the stats season and 78.0% with one at the following 1 July (the missing
fifth is mostly the current season, which has no "next July" yet); coverage ≥ 0.95 in every
league-season. Log10 value has sd ≈ 0.56 in every role (a factor of ~3.6); the raw relationships
are the expected ones — a rise of ~1.25 log10 across the contribution bins, an age curve peaking
at 24–30 and falling by 0.7 log10 after 34, a minutes gradient, and league premia (Premier League
7.06 vs Ligue 1 6.51 in log10 — about ×3.5).

**Target: log value at the 1 July *after* the stats season.** `value_july` is dated at the start
of the season, so it prices the *previous* season's profile; the backtest and the resale model
need the price the market puts on a season once it has happened, which is the next 1 July. The
season-of-stats value stays as a feature candidate (the market's prior).

**Club tier: the club's Elo on 1 July, not its squad-value rank.** After contribution, age, role,
league and season (R² 0.375), squad-value rank explains 38% of the remaining variance, Elo 28%,
expected points 22% — but the rank contains the player's own value and is the same market
judging itself, so it would leak the target into a feature. Elo is an independent measure of the
club's strength. Rejected: squad-value rank (circular), expected points (weaker and it only
exists for the Big 5 — the candidate pool needs a tier for feeder clubs too).

## Step 2 — Market model family (the open choice in spec §4.D)

Target: log10 value at the 1 July after the stats season. Features: contribution (shrunk point),
recency history, minutes, age, role, league, club Elo on that 1 July, season, and the market's
prior (log value at the start of the season). Two families, both leave-future-out (train on
seasons ≤ s−1, test on s, 2016-17 → 2024-25): (a) an OLS regression with role × (contribution,
age, age²) and categorical league/season; (b) `HistGradientBoostingRegressor` with monotone
constraints (value rises with contribution, history, minutes, Elo and the prior; age unconstrained).
Kill check: held-out error (RMSE/MAE in log10, median absolute percentage error in euros) and a
monotonicity sanity table; the lower error among families that pass; ties to the simpler.

In [6]:
from sklearn.ensemble import HistGradientBoostingRegressor
import statsmodels.formula.api as smf

# club Elo on the 1 July after the season, for the club the player was at
elo_next = []
for (comp, season, club_id), _ in valued.groupby(["competition_id", "season", "club_id"]):
    name = elo_names.get(club_id)
    if name is None:
        continue
    elo_next.append((comp, season, club_id, elo_panel.elo_on_dates(clubelo.fetch_club(name), pd.Series([f"{season + 1}-07-01"]))[0]))
elo_next = pd.DataFrame(elo_next, columns=["competition_id", "season", "club_id", "club_elo_next"])
m = valued.merge(elo_next, on=["competition_id", "season", "club_id"], how="left").dropna(subset=["value_next_july", "club_elo_next"]).copy()
m["y"] = np.log10(m.value_next_july)
m["log_prior"] = np.log10(m.value_july)
m["history_point"] = m.history_point.fillna(m.point)
m["elo_c"] = (m.club_elo_next - 1600) / 100
print(len(m), "rows with target, prior, Elo |", m.season.min(), "-", m.season.max())

FORMULA = "y ~ C(role) * (point + age + I(age**2)) + history_point + np.log(minutes) + elo_c + log_prior + C(competition_id) + C(season)"
NUM = ["point", "history_point", "minutes", "age", "elo_c", "log_prior"]
CAT = ["role", "competition_id"]


def gbm_frame(frame):
    X = frame[NUM].copy()
    for c in CAT:
        X[c] = pd.Categorical(frame[c], categories=sorted(m[c].unique())).codes
    return X


MONO = [1, 1, 1, 0, 1, 1, 0, 0]  # point, history, minutes, age, elo, prior, role, league


def fit_gbm(train):
    return HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=40, monotonic_cst=MONO, categorical_features=[6, 7], random_state=0).fit(gbm_frame(train), train.y)


results = []
for s in range(2016, 2025):
    train, test = m[m.season < s], m[m.season == s]
    if len(test) == 0:
        continue
    ols = smf.ols(FORMULA, data=train).fit()
    test_ols = test.copy(); test_ols["season"] = train.season.max()  # the season effect is unknown for a new season: use the latest known
    p_ols = ols.predict(test_ols)
    gbm = fit_gbm(train); p_gbm = gbm.predict(gbm_frame(test))
    for name, pred in [("ols", p_ols), ("gbm", p_gbm), ("prior only", test.log_prior)]:
        err = test.y - pred
        results.append({"season": s, "family": name, "n": len(test), "rmse": np.sqrt((err ** 2).mean()), "mae": err.abs().mean(), "mdape_eur": np.median(np.abs(10 ** pred / test.value_next_july - 1))})
res = pd.DataFrame(results)
print(res.pivot_table(index="season", columns="family", values="rmse").round(3).to_string())
print("\naverages over held-out seasons:"); print(res.groupby("family")[["rmse", "mae", "mdape_eur"]].mean().round(3).to_string())

16560 rows with target, prior, Elo | 2014 - 2024


family    gbm    ols  prior only
season                          
2016    0.182  0.193       0.336
2017    0.222  0.222       0.381
2018    0.229  0.206       0.379
2019    0.172  0.212       0.264
2020    0.181  0.192       0.350
2021    0.153  0.163       0.279
2022    0.175  0.182       0.320
2023    0.177  0.190       0.352
2024    0.173  0.188       0.336

averages over held-out seasons:
             rmse    mae  mdape_eur
family                             
gbm         0.185  0.138      0.240
ols         0.194  0.147      0.275
prior only  0.333  0.219      0.328


In [7]:
# Monotonicity sanity: shift one feature at a time on the last held-out season, everything else held
train, test = m[m.season < 2024], m[m.season == 2024]
ols = smf.ols(FORMULA, data=train).fit(); gbm = fit_gbm(train)
base_t = test.copy(); base_t["season"] = 2023
checks = {"point +0.1": ("point", 0.1), "history +0.1": ("history_point", 0.1), "minutes +500": ("minutes", 500), "elo +100": ("elo_c", 1.0), "prior ×2": ("log_prior", np.log10(2)), "age 24→28": ("age", 4), "age 28→32": ("age", 4)}
rows = []
for label, (col, delta) in checks.items():
    a = base_t.copy(); b = base_t.copy()
    if label.startswith("age"):
        start = int(label.split()[1].split("→")[0]); a["age"] = start; b["age"] = start + delta
    else:
        b[col] = b[col] + delta
    d_ols = (ols.predict(b) - ols.predict(a)).mean(); d_gbm = (gbm.predict(gbm_frame(b)) - gbm.predict(gbm_frame(a))).mean()
    share_up = ((gbm.predict(gbm_frame(b)) - gbm.predict(gbm_frame(a))) >= 0).mean()
    rows.append((label, round(d_ols, 3), round(d_gbm, 3), round(share_up, 2)))
print(pd.DataFrame(rows, columns=["shift", "Δ log10 OLS", "Δ log10 GBM", "GBM share of rows non-decreasing"]).to_string(index=False))
print("\n(expected: positive for point/history/minutes/elo/prior; age 24→28 ≈ 0 or slightly positive, 28→32 negative)")

       shift  Δ log10 OLS  Δ log10 GBM  GBM share of rows non-decreasing
  point +0.1        0.051        0.047                               1.0
history +0.1        0.016        0.006                               1.0
minutes +500        0.058        0.059                               1.0
    elo +100        0.102        0.067                               1.0
    prior ×2        0.153        0.210                               1.0
   age 24→28       -0.179       -0.109                               0.0
   age 28→32       -0.202       -0.176                               0.0

(expected: positive for point/history/minutes/elo/prior; age 24→28 ≈ 0 or slightly positive, 28→32 negative)


### Step 2 note — market model family, from the two tables above

**Chosen: gradient boosting with monotone constraints** (`sklearn` `HistGradientBoostingRegressor`,
400 trees, 15 leaves, min 40 rows per leaf; value constrained to rise with contribution, history,
minutes, club Elo and the market's prior; age and the categoricals free). Leave-future-out over
nine seasons (2016-17 → 2024-25, 16,560 rows): RMSE 0.185 log10 against 0.194 for the OLS
regression (better in 8 of 9 seasons) and 0.333 for the market's own prior alone; median error in
euros 24% (OLS 28%, prior 33%). Both families pass the sanity table — a +0.1 in contribution per
90 adds 0.04–0.05 log10 (≈ +11%), +100 club Elo adds 0.07–0.10, doubling the prior adds 0.15–0.21;
the market already discounts age from 24 (−0.11 to −0.18 log10 for 24 → 28, more after 28), which
is a resale-horizon effect, not a violation. Rejected: OLS (higher error; kept as the
interpretable check in the notebook), LightGBM (no need — sklearn's boosting suffices), any
model without the monotone constraints (the sanity table is a requirement, not a hope).
Ported: `scout.models.market` (`prepare`, `features`, `fit`, `leave_future_out`).

### Step 2 check — `scout.models.market` reproduces the held-out table

In [8]:
from scout.models import market as market_model

prepared = market_model.prepare(valued.merge(elo_next, on=["competition_id", "season", "club_id"], how="left"))
leagues = sorted(prepared.competition_id.unique())
lfo = market_model.leave_future_out(prepared, leagues)
print(lfo.round(3).to_string(index=False))
print("average rmse:", round(lfo.rmse.mean(), 3), "| mae:", round(lfo.mae.mean(), 3), "| mdape:", round(lfo.mdape_eur.mean(), 3), "(above: 0.185 / 0.138 / 0.239)")

 season    n  rmse   mae  mdape_eur
   2016 1507 0.182 0.134      0.226
   2017 1530 0.222 0.166      0.264
   2018 1498 0.229 0.166      0.265
   2019 1499 0.172 0.135      0.276
   2020 1549 0.181 0.138      0.248
   2021 1540 0.153 0.116      0.207
   2022 1503 0.175 0.131      0.237
   2023 1491 0.177 0.131      0.217
   2024 1467 0.173 0.128      0.219
average rmse: 0.185 | mae: 0.138 | mdape: 0.24 (above: 0.185 / 0.138 / 0.239)


## Step 3 — Price gaps

The residual (actual − expected log value, held-out) is the market's disagreement with the model.
Does it persist — for the same player year to year, by club, by league, by age band? A persistent
component enters the Phase 5 "market's own ranking" baseline and the writeup; noise is reported as
noise.

In [9]:
gaps = []
for s in range(2016, 2025):
    train, test = prepared[prepared.season < s], prepared[prepared.season == s]
    pred = market_model.fit(train, leagues).predict(market_model.features(test, leagues))
    gaps.append(test.assign(gap=test.y - pred))
gaps = pd.concat(gaps)
print(len(gaps), "held-out rows | gap sd", round(gaps.gap.std(), 3), "| share |gap| > 0.3 log10 (×2):", f"{(gaps.gap.abs() > 0.3).mean():.1%}")
nxt = gaps.assign(season=gaps.season - 1)[["tm_player_id", "season", "gap"]].rename(columns={"gap": "gap_next"})
pg = gaps.merge(nxt, on=["tm_player_id", "season"])
print(f"same player, next season: r = {pg.gap.corr(pg.gap_next):.3f} (n={len(pg)})")
by_age = gaps.groupby(pd.cut(gaps.age, [16, 21, 24, 27, 30, 45]), observed=True).gap.agg(["mean", "size"]).round(3)
print("\nmean gap by age band:"); print(by_age.T.to_string())
print("\nmean gap by league:", gaps.groupby("competition_id").gap.mean().round(3).to_dict())
print("mean gap by role:", gaps.groupby("role").gap.mean().round(3).to_dict())
# clubs: is a club's mean gap in seasons ≤ s informative about its gap in s+1?
club_gap = gaps.groupby(["club_id", "season"]).gap.agg(["mean", "size"]).reset_index()
club_gap = club_gap[club_gap["size"] >= 5]
cn = club_gap.assign(season=club_gap.season - 1)[["club_id", "season", "mean"]].rename(columns={"mean": "next"})
cp = club_gap.merge(cn, on=["club_id", "season"])
print(f"\nclub mean gap (≥5 players), year to year: r = {cp['mean'].corr(cp['next']):.3f} (n={len(cp)} club-seasons)")
print("clubs with the most persistently under-priced players (mean gap over all held-out seasons, ≥20 rows):")
club_names = tm_clubs.drop_duplicates("club_id").set_index("club_id").club_name
top = gaps.groupby("club_id").gap.agg(["mean", "size"]); top = top[top["size"] >= 20].sort_values("mean")
print(top.head(8).assign(club=lambda d: d.index.map(club_names)).round(3).to_string())

13584 held-out rows | gap sd 0.184 | share |gap| > 0.3 log10 (×2): 9.6%
same player, next season: r = 0.052 (n=9661)

mean gap by age band:
age   (16, 21]  (21, 24]  (24, 27]  (27, 30]  (30, 45]
mean     0.096      0.06     0.027      0.01     0.007
size   882.000   3054.00  3815.000   3174.00  2656.000

mean gap by league: {'ES1': 0.037, 'FR1': 0.058, 'GB1': 0.009, 'IT1': 0.018, 'L1': 0.036}
mean gap by role: {'CB': 0.033, 'CM': 0.029, 'FB': 0.031, 'ST': 0.024, 'W': 0.036}

club mean gap (≥5 players), year to year: r = 0.121 (n=663 club-seasons)
clubs with the most persistently under-priced players (mean gap over all held-out seasons, ≥20 rows):
          mean  size                        club
club_id                                         
10.0    -0.104    21           Arminia Bielefeld
1005.0  -0.037    58                    US Lecce
985.0   -0.034   163           Manchester United
12.0    -0.029   159  Associazione Sportiva Roma
281.0   -0.027   164             Manchester City
35